# Đánh giá kết quả chạy thử framework `SK2Decompile`

## Khởi tạo thư mục & Cài đặt trình biên dịch / thư viện

In [1]:
import os

os.makedirs("/kaggle/working/outputs", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)
os.makedirs("/kaggle/tmp/eval_sandbox", exist_ok=True)

REPOS_DIR = "/kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos"
EVAL_DIR = f"{REPOS_DIR}/LLM4Decompile/sk2decompile/evaluation"

# Kiểm tra các script đánh giá có sẵn trong repo
!ls -la {EVAL_DIR}

# Cài đặt thư viện & trình biên dịch C
!pip install -q tree-sitter==0.24.0 tree-sitter-c==0.23.4 pandas numpy tqdm editdistance
!apt-get update -qq && apt-get install -y -qq gcc clang make

total 48
drwxrwxrwx 1 root root    0 Nov 21 01:57 bringupbench
drwxrwxrwx 1 root root    0 Nov 21 01:57 data
-rw-rw-rw- 1 root root 3259 Nov 20 03:14 evaluate_exe.py
-rw-rw-rw- 1 root root 4413 Nov 20 03:14 evaluate_r2i.py
-rw-rw-rw- 1 root root 9525 Nov 20 03:14 gpt_judge.py
-rw-rw-rw- 1 root root 6974 Nov 20 03:14 inf_type.py
-rw-rw-rw- 1 root root 3300 Nov 20 03:14 llm_server.py
drwxrwxrwx 1 root root    0 Nov 21 01:57 metrics
-rw-rw-rw- 1 root root 6632 Nov 20 03:14 normalize_pseudo.py
-rw-rw-rw- 1 root root 7621 Nov 20 03:14 sk2decompile_inf.py
-rw-rw-rw- 1 root root 2119 Nov 20 03:14 stack.yaml
-rw-rw-rw- 1 root root 2619 Nov 20 03:14 template.txt
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 125687 files and directories currently installed.)
Preparing to unpack .../libc6-dev_2.39-0ubuntu8.

## Chạy script đánh giá thực thi & khả năng đọc hiểu (R2I)

In [2]:
import os
import re
import json
import tempfile
import subprocess
from multiprocessing import Pool
from tqdm import tqdm

INPUT_JSONL = "/kaggle/input/datasets/wood1enqxyn/sk2-outputs/outputs/sk2_recovered_code_c.jsonl"
OUTPUT_JSONL = "/kaggle/tmp/sk2_full_eval.jsonl"

STD_HEADERS = """
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <stdbool.h>
#include <stddef.h>
#include <string.h>
#include <math.h>
#include <limits.h>
#include <ctype.h>
"""

def extract_c_code(text):
    if not text:
        return ""
    # Khôi phục ký tự khoảng trắng / xuống dòng bị lỗi detokenizer của vLLM
    text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
    m = re.search(r"```(?:c|cpp)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

def restore_func_name(code, target_name):
    """Khôi phục tên hàm giống logic evaluate_exe.py gốc (an toàn hơn với regex)"""
    if not code or not target_name or "(" not in code:
        return code
    header, rest = code.split("(", 1)
    tokens = header.split()
    old_name = tokens[-1].lstrip("*") if tokens else ""
    if old_name and old_name != target_name and re.match(r"^[A-Za-z_]\w*$", old_name):
        header = re.sub(rf"\b{re.escape(old_name)}\b(?!.*\b{re.escape(old_name)}\b)", target_name, header)
        return header + "(" + rest
    return code

def eval_single_sample(record):
    clean_p1 = extract_c_code(record.get("phase1_ir", ""))
    code = extract_c_code(record.get("recovered_code", ""))
    code = restore_func_name(code, record.get("func_name", ""))

    # Ghi đè bản đã làm sạch để Cell 4 dùng luôn
    record["phase1_ir"] = clean_p1
    record["recovered_code"] = code

    func_dep = str(record.get("func_dep", "") or "").strip()
    if func_dep.endswith(".c") or func_dep.endswith(".h") or ("\n" not in func_dep and "/" in func_dep):
        func_dep = ""

    func_test = str(record.get("test", "") or "").strip()
    has_test = len(func_test) > 0

    flag_comp = False
    flag_exe = False

    if not code:
        record["comp_ok"] = False
        record["exe_ok"] = False
        record["r2i_score"] = 0.0
        return record

    full_src = f"{STD_HEADERS}\n{func_dep}\n{code}\n" + (f"\n{func_test}\n" if has_test else "")

    with tempfile.TemporaryDirectory() as temp_dir:
        c_file = os.path.join(temp_dir, "prog.c")
        bin_file = os.path.join(temp_dir, "prog")
        with open(c_file, "w", encoding="utf-8") as f:
            f.write(full_src)

        if has_test:
            cmd = ["gcc", "-O0", c_file, "-o", bin_file, "-lm"]
        else:
            cmd = ["gcc", "-O0", "-c", c_file, "-o", bin_file + ".o"]

        try:
            subprocess.run(cmd, check=True, timeout=10, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            flag_comp = True
        except Exception:
            flag_comp = False

        if flag_comp and has_test:
            try:
                subprocess.run([bin_file], check=True, timeout=10, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                flag_exe = True
            except Exception:
                flag_exe = False

    record["comp_ok"] = flag_comp
    record["exe_ok"] = flag_exe
    record["r2i_score"] = 0.0
    return record

# Đọc dữ liệu đầu vào (hỗ trợ cả JSONL và mảng JSON)
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    content = f.read().strip()
    if content.startswith("["):
        records = json.loads(content)
    else:
        records = [json.loads(line) for line in content.splitlines() if line.strip()]

print(f"Bắt đầu đánh giá {len(records)} mẫu bằng 4 tiến trình...")
with Pool(processes=4) as pool:
    results = list(tqdm(pool.imap(eval_single_sample, records), total=len(records)))

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f_out:
    for r in results:
        f_out.write(json.dumps(r, ensure_ascii=False) + "\n")

comp_rate = sum(r["comp_ok"] for r in results) / len(results) * 100
exe_rate = sum(r["exe_ok"] for r in results) / len(results) * 100
has_test_cnt = sum(1 for r in results if str(r.get("test", "") or "").strip())
print(f"Hoàn tất! Số mẫu có unit test: {has_test_cnt}/{len(results)}")
print(f"Re-compilation: {comp_rate:.2f}% | Re-executability: {exe_rate:.2f}%")

Bắt đầu đánh giá 880 mẫu bằng 4 tiến trình...


100%|██████████| 880/880 [00:17<00:00, 50.47it/s]


Hoàn tất! Số mẫu có unit test: 0/880
Re-compilation: 15.45% | Re-executability: 0.00%


## Tổng hợp `sk2_metrics_summary.csv` & Phân loại lỗi 2 pha (`sk2_error_classification.json`)

In [3]:
import os
import re
import json
import subprocess
import tempfile
import pandas as pd
import editdistance

EVAL_RESULT_PATH = "/kaggle/tmp/sk2_full_eval.jsonl"
CSV_OUT = "/kaggle/working/outputs/sk2_metrics_summary.csv"
ERROR_JSON_OUT = "/kaggle/working/logs/sk2_error_classification.json"

os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)
os.makedirs(os.path.dirname(ERROR_JSON_OUT), exist_ok=True)

records = []
with open(EVAL_RESULT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert len(records) > 0, f"LỖI: File {EVAL_RESULT_PATH} rỗng!"

STD_HEADERS = """
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <stdbool.h>
#include <stddef.h>
#include <string.h>
#include <math.h>
#include <limits.h>
#include <ctype.h>
"""

def extract_c_code(text):
    if not text:
        return ""
    text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
    m = re.search(r"```(?:c|cpp)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

# Hàm tính Edit Similarity (%)
def calc_edit_sim(pred, ref):
    if not pred and not ref:
        return 100.0
    max_len = max(len(pred), len(ref), 1)
    return (1.0 - editdistance.eval(pred, ref) / max_len) * 100.0

# Hàm phân loại lỗi biên dịch qua stderr của gcc
def classify_gcc_error(c_code, func_dep=""):
    clean_code = extract_c_code(c_code)
    if not clean_code:
        return "empty_code"

    full_code = f"{STD_HEADERS}\n{func_dep}\n{clean_code}\n"
    c_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".c", mode="w", encoding="utf-8", delete=False) as tf:
            tf.write(full_code)
            c_path = tf.name
        proc = subprocess.run(
            ["gcc", "-fsyntax-only", c_path],
            capture_output=True, text=True, timeout=5
        )
    except subprocess.TimeoutExpired:
        return "other_compile_error"
    finally:
        if c_path and os.path.exists(c_path):
            os.remove(c_path)

    if proc.returncode == 0:
        return "none"
    err = proc.stderr.lower()
    if "expected" in err or "syntax error" in err or "stray" in err:
        return "syntax_error"
    elif "incompatible type" in err or "undeclared" in err or "conflicting types" in err or "unknown type" in err:
        return "type_or_symbol_error"
    else:
        return "other_compile_error"

error_stats = {
    "phase1_syntax_errors": 0,
    "phase1_type_errors": 0,
    "phase2_syntax_errors": 0,
    "phase2_type_errors": 0,
    "propagated_errors_p1_to_p2": 0, # Lỗi từ Pha 1 kéo sang Pha 2
    "fixed_by_phase2": 0,             # Pha 1 lỗi nhưng Pha 2 tự sửa được
    "logic_runtime_errors": 0,        # Biên dịch được nhưng Fail Unit Test
    "error_examples": []
}

rows_for_df = []
for idx, r in enumerate(records):
    opt = r.get("opt", "unknown")
    comp_ok = bool(r.get("comp_ok", False))
    exe_ok = bool(r.get("exe_ok", False))
    r2i_score = float(r.get("r2i_score", 0.0))
    has_test = len(str(r.get("test", "") or "").strip()) > 0

    ref_code = r.get("c_func", "") or r.get("func", "")
    pred_code = extract_c_code(r.get("recovered_code", ""))
    p1_code = extract_c_code(r.get("phase1_ir", ""))

    func_dep = str(r.get("func_dep", "") or "").strip()
    if func_dep.endswith(".c") or func_dep.endswith(".h") or ("\n" not in func_dep and "/" in func_dep):
        func_dep = ""

    es = calc_edit_sim(pred_code, ref_code)
    rows_for_df.append({
        "opt": opt,
        "re_compilation": int(comp_ok) * 100,
        "re_executability": int(exe_ok) * 100,
        "r2i_readability": r2i_score,
        "edit_similarity": es
    })

    # Phân loại lỗi cho Pha 1 vs Pha 2
    err_p1 = classify_gcc_error(p1_code, func_dep) if p1_code else "none"
    err_p2 = "none" if comp_ok else classify_gcc_error(pred_code, func_dep)

    if err_p1 == "syntax_error": error_stats["phase1_syntax_errors"] += 1
    elif err_p1 != "none": error_stats["phase1_type_errors"] += 1

    if err_p2 == "syntax_error": error_stats["phase2_syntax_errors"] += 1
    elif err_p2 != "none": error_stats["phase2_type_errors"] += 1

    if err_p1 != "none" and err_p2 != "none":
        error_stats["propagated_errors_p1_to_p2"] += 1
    elif err_p1 != "none" and comp_ok:
        error_stats["fixed_by_phase2"] += 1

    if has_test and comp_ok and not exe_ok:
        error_stats["logic_runtime_errors"] += 1

    if (err_p2 != "none" or (has_test and not exe_ok)) and len(error_stats["error_examples"]) < 20:
        error_stats["error_examples"].append({
            "sample_idx": idx,
            "opt": opt,
            "p1_error": err_p1,
            "p2_error": err_p2,
            "exe_ok": exe_ok
        })

# Xuất bảng tổng hợp CSV theo từng mức tối ưu (-O0 -> -O3) + Trung bình
df = pd.DataFrame(rows_for_df)
summary_df = df.groupby("opt").mean().reset_index()
overall = pd.DataFrame([{
    "opt": "ALL_AVG",
    "re_compilation": df["re_compilation"].mean(),
    "re_executability": df["re_executability"].mean(),
    "r2i_readability": df["r2i_readability"].mean(),
    "edit_similarity": df["edit_similarity"].mean()
}])
summary_df = pd.concat([summary_df, overall], ignore_index=True).round(2)
summary_df.to_csv(CSV_OUT, index=False)

with open(ERROR_JSON_OUT, "w", encoding="utf-8") as f_err:
    json.dump(error_stats, f_err, ensure_ascii=False, indent=2)

print("=== BẢNG CHỈ SỐ SK2DECOMPILE ===")
print(summary_df)
print("\n=== THỐNG KÊ LỖI 2 PHA ===")
print({k: v for k, v in error_stats.items() if k != "error_examples"})

=== BẢNG CHỈ SỐ SK2DECOMPILE ===
       opt  re_compilation  re_executability  r2i_readability  edit_similarity
0       O0           22.07               0.0              0.0            31.22
1       O1            8.29               0.0              0.0            30.04
2       O2           13.00               0.0              0.0            30.42
3       O3           17.84               0.0              0.0            26.35
4  ALL_AVG           15.45               0.0              0.0            29.42

=== THỐNG KÊ LỖI 2 PHA ===
{'phase1_syntax_errors': 136, 'phase1_type_errors': 598, 'phase2_syntax_errors': 183, 'phase2_type_errors': 561, 'propagated_errors_p1_to_p2': 727, 'fixed_by_phase2': 7, 'logic_runtime_errors': 0}


## R2I cloning

In [4]:
!rm -rf /kaggle/working/evaluation
!cp -r /kaggle/input/datasets/wood1enqxyn/sk2decompile-autodecomp-repos/LLM4Decompile/sk2decompile/evaluation /kaggle/working/evaluation
!git clone https://github.com/e0mh4/R2I.git /kaggle/working/evaluation/metrics/R2I
!ls -la /kaggle/working/evaluation/metrics/R2I
!cat /kaggle/working/evaluation/metrics/R2I/requirements.txt 2>/dev/null || true

Cloning into '/kaggle/working/evaluation/metrics/R2I'...
remote: Enumerating objects: 956, done.
remote: Counting objects: 100% (956/956), done.
remote: Compressing objects: 100% (336/336), done.
remote: Total 956 (delta 642), reused 863 (delta 606), pack-reused 0 (from 0)
Receiving objects: 100% (956/956), 2.01 MiB | 13.02 MiB/s, done.
Resolving deltas: 100% (642/642), done.
total 36
drwxr-xr-x 7 root root 4096 Nov 21 01:59 .
drwxr-xr-x 3 root root 4096 Nov 21 01:59 ..
drwxr-xr-x 3 root root 4096 Nov 21 01:59 dataset
drwxr-xr-x 3 root root 4096 Nov 21 01:59 eval
drwxr-xr-x 8 root root 4096 Nov 21 01:59 .git
drwxr-xr-x 4 root root 4096 Nov 21 01:59 pycparser
-rwxr-xr-x 1 root root 2534 Nov 21 01:59 README.md
-rwxr-xr-x 1 root root 1309 Nov 21 01:59 run.sh
drwxr-xr-x 2 root root 4096 Nov 21 01:59 src


## Cài thư viện phụ thuộc của R2I

In [5]:
# Cài requirements của R2I (nếu có)
!pip install -q -r /kaggle/working/evaluation/metrics/R2I/requirements.txt 2>/dev/null || true

# Tạo sẵn thư mục để evaluate_r2i.py không bị lỗi khi gọi shutil.rmtree
!mkdir -p /kaggle/working/evaluation/metrics/R2I/dataset/test
!mkdir -p /kaggle/working/outputs/r2i_results

## Chuyển đổi `sk2_recovered_code_c.jsonl` sang cấu trúc `.json + .c + .h`

In [6]:
import os
import re
import json

EVAL_WORK_DIR = "/kaggle/working/evaluation"
INPUT_JSONL = "/kaggle/input/datasets/wood1enqxyn/sk2-outputs/outputs/sk2_recovered_code_c.jsonl"
DATASET_NAME = "sk2_eval"
JSON_OUT = f"{EVAL_WORK_DIR}/data/{DATASET_NAME}.json"

STD_HEADERS = """#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <stdbool.h>
#include <stddef.h>
#include <string.h>
#include <math.h>
#include <limits.h>
#include <ctype.h>
"""

def clean_code(text):
    if not text:
        return ""
    text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
    m = re.search(r"```(?:c|cpp)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

json_array = []
for idx, r in enumerate(records):
    opt = r.get("opt", "O0").lstrip("-")  # chuẩn hóa về O0, O1, O2, O3
    func_name = r.get("func_name", f"func_{idx}")
    sample_idx = r.get("index", idx)

    func_dep = str(r.get("func_dep", "") or "").strip()
    if func_dep.endswith(".c") or func_dep.endswith(".h") or ("\n" not in func_dep and "/" in func_dep):
        func_dep = ""
    header_content = STD_HEADERS + ("\n" + func_dep if func_dep else "")

    json_array.append({
        "index": sample_idx,
        "opt": opt,
        "language": "c",
        "func_name": func_name,
        "func": r.get("c_func", "") or r.get("func", "")
    })

    # Xuất file .c và .h cho cả 2 pha để R2I so sánh chéo
    pairs = [
        ("sk2_phase1", clean_code(r.get("phase1_ir", ""))),
        ("sk2_phase2", clean_code(r.get("recovered_code", "")))
    ]
    for decompiler, code in pairs:
        c_dir = f"{EVAL_WORK_DIR}/model_outputs/{DATASET_NAME}/{decompiler}/{opt}"
        h_dir = f"{EVAL_WORK_DIR}/headers/{DATASET_NAME}/{decompiler}/{opt}"
        os.makedirs(c_dir, exist_ok=True)
        os.makedirs(h_dir, exist_ok=True)

        with open(f"{c_dir}/{sample_idx}_{opt}.c", "w", encoding="utf-8") as fc:
            fc.write(code)
        with open(f"{h_dir}/{sample_idx}_{opt}.h", "w", encoding="utf-8") as fh:
            fh.write(header_content)

os.makedirs(os.path.dirname(JSON_OUT), exist_ok=True)
with open(JSON_OUT, "w", encoding="utf-8") as f_out:
    json.dump(json_array, f_out, ensure_ascii=False)

print(f"Đã xuất xong {len(json_array)} mẫu vào {JSON_OUT} và các thư mục model_outputs/headers.")

Đã xuất xong 880 mẫu vào /kaggle/working/evaluation/data/sk2_eval.json và các thư mục model_outputs/headers.


## Chạy evaluate_r2i.py và xem kết quả

In [7]:
!pip install -q ply

run_py_code = '''import os
import glob
import shutil
import subprocess
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

BASE = os.path.dirname(os.path.abspath(__file__))

def _extract_one(args):
    d, experiment, filename = args
    cwd = os.path.join(BASE, "dataset", experiment, d)
    cmd = ["python3", os.path.join(BASE, "src", "extractor.py"), "-s", "-p", f"{experiment}/{d}", filename]
    subprocess.run(cmd, cwd=cwd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def run_r2i(decompilers, names=None, experiment="test", weight_version="paper"):
    extract_root = os.path.join(BASE, "extract")
    eval_dir = os.path.join(BASE, "eval", experiment)
    bin_dir = os.path.join(eval_dir, "bin")

    # Dọn kết quả cũ của opt trước
    shutil.rmtree(extract_root, ignore_errors=True)
    shutil.rmtree(bin_dir, ignore_errors=True)
    os.makedirs(bin_dir, exist_ok=True)

    # 1. Chạy extractor.py song song cho tất cả file .c
    tasks = []
    for d in decompilers:
        os.makedirs(os.path.join(extract_root, experiment, d), exist_ok=True)
        c_dir = os.path.join(BASE, "dataset", experiment, d, "c")
        if not os.path.exists(c_dir):
            continue
        for fname in sorted(os.listdir(c_dir)):
            if fname.endswith(".c"):
                tasks.append((d, experiment, fname))

    with ProcessPoolExecutor(max_workers=4) as ex:
        list(ex.map(_extract_one, tasks))

    # 2. Chạy aggregator.py
    agg_cmd = ["python3", os.path.join(BASE, "src", "aggregator.py"), experiment, "-s", "-t", *decompilers]
    subprocess.run(agg_cmd, cwd=extract_root, check=False)

    # 3. Chạy evaluator.py
    eval_cmd = ["python3", os.path.join(BASE, "src", "evaluator.py"), "-t", *decompilers, "-w", weight_version]
    subprocess.run(eval_cmd, cwd=eval_dir, check=False)

    # 4. Đọc điểm R2I trực tiếp từ eval/test/bin/*.csv (bỏ qua mean.py bị hardcode 6 decompiler)
    csv_files = glob.glob(os.path.join(bin_dir, "*.csv"))
    if not csv_files:
        return {d: 0.0 for d in decompilers}

    dfs = [pd.read_csv(f) for f in csv_files]
    data = pd.concat(dfs, axis=0, ignore_index=True)

    scores = {}
    for d in decompilers:
        sub = data[data["decompiler"] == d]["r2i"]
        scores[d] = float(sub.mean()) if len(sub) > 0 else 0.0

    # Dọn dẹp tạm
    shutil.rmtree(extract_root, ignore_errors=True)
    shutil.rmtree(bin_dir, ignore_errors=True)
    return scores
'''

with open("/kaggle/working/evaluation/metrics/R2I/run.py", "w", encoding="utf-8") as f:
    f.write(run_py_code)

print("Đã tạo xong /kaggle/working/evaluation/metrics/R2I/run.py")

Đã tạo xong /kaggle/working/evaluation/metrics/R2I/run.py


In [8]:
!cd /kaggle/working/evaluation && PYTHONPATH=. python evaluate_r2i.py \
    --json_file ./data/sk2_eval.json \
    --decompilers sk2_phase1,sk2_phase2 \
    --output_path /kaggle/working/outputs/r2i_results

# Xem các file kết quả R2I vừa xuất ra
!ls -la /kaggle/working/outputs/r2i_results

/kaggle/working/evaluation/metrics/R2I/src/extractor.py:139: SyntaxWarning: invalid escape sequence '\s'
  score['lol'] = max(len(re.split('[\s,:;\(\)\{\}]+', line.split('//', 1)[0].split('/*', 1)[0].strip())) for line in lines)
/kaggle/working/evaluation/metrics/R2I/src/evaluator.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.032258065' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.W[dv[0]] = dv[w]
[+] Successfully sieved common functions...
[+] Successfully saved feature & r2i values: r2i.csv
[+] Successfully saved evaluation on each binary
/kaggle/working/evaluation/metrics/R2I/src/evaluator.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.032258065' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.W[dv[0]] = dv[w]

## Update statistic

In [9]:
import os
import re
import json
import pandas as pd

CSV_PATH = "/kaggle/working/outputs/sk2_metrics_summary.csv"
R2I_TXT = "/kaggle/working/outputs/r2i_results/r2i_results_results.txt"
EVAL_JSONL = "/kaggle/tmp/sk2_full_eval.jsonl"

# 1. Đọc file kết quả R2I (tự lấy lượt chạy mới nhất nếu file bị append nhiều lần)
r2i_p1, r2i_p2 = {}, {}
with open(R2I_TXT, "r", encoding="utf-8") as f:
    for line in f:
        for opt, dec, val in re.findall(r"r2i\s+(O[0-3])\s+([\w\-]+):\s*([0-9.]+)", line):
            if dec == "sk2_phase1":
                r2i_p1[opt] = float(val)
            elif dec == "sk2_phase2":
                r2i_p2[opt] = float(val)

assert r2i_p2, f"Chưa tìm thấy điểm sk2_phase2 trong {R2I_TXT}! Hãy kiểm tra lại kết quả Cell chạy R2I."

# 2. Lấy số lượng mẫu theo từng opt để tính trung bình ALL_AVG có trọng số chính xác
opt_counts = {}
if os.path.exists(EVAL_JSONL):
    with open(EVAL_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line).get("opt", "").lstrip("-")
                opt_counts[o] = opt_counts.get(o, 0) + 1

def weighted_avg(score_dict):
    valid = [o for o in ["O0", "O1", "O2", "O3"] if o in score_dict]
    if not valid:
        return 0.0
    if opt_counts and all(opt_counts.get(o, 0) > 0 for o in valid):
        total = sum(opt_counts[o] for o in valid)
        return round(sum(score_dict[o] * opt_counts[o] for o in valid) / total, 2)
    return round(sum(score_dict[o] for o in valid) / len(valid), 2)

# 3. Gộp vào bảng sk2_metrics_summary.csv
df = pd.read_csv(CSV_PATH)
df["opt"] = df["opt"].astype(str).str.lstrip("-")

# Cập nhật cột r2i_readability (lấy điểm Pha 2 - mã phục hồi cuối cùng)
df["r2i_readability"] = df["opt"].map(lambda o: r2i_p2.get(o, weighted_avg(r2i_p2) if o == "ALL_AVG" else 0.0))

# Thêm luôn cột r2i_phase1 ngay bên cạnh để tiện so sánh mức cải thiện Pha 1 -> Pha 2
if r2i_p1:
    p1_vals = df["opt"].map(lambda o: r2i_p1.get(o, weighted_avg(r2i_p1) if o == "ALL_AVG" else 0.0))
    if "r2i_phase1" in df.columns:
        df["r2i_phase1"] = p1_vals
    else:
        insert_idx = df.columns.get_loc("r2i_readability")
        df.insert(insert_idx, "r2i_phase1", p1_vals)

df = df.round(2)
df.to_csv(CSV_PATH, index=False)

print("=== BẢNG CHỈ SỐ SK2DECOMPILE (ĐÃ CẬP NHẬT R2I) ===")
print(df)

=== BẢNG CHỈ SỐ SK2DECOMPILE (ĐÃ CẬP NHẬT R2I) ===
       opt  re_compilation  re_executability  r2i_phase1  r2i_readability  \
0       O0           22.07               0.0        6.49             5.38   
1       O1            8.29               0.0       13.53            10.85   
2       O2           13.00               0.0        4.94             2.47   
3       O3           17.84               0.0        5.15             4.76   
4  ALL_AVG           15.45               0.0        7.51             5.90   

   edit_similarity  
0            31.22  
1            30.04  
2            30.42  
3            26.35  
4            29.42  
